In [1]:
!pip install evaluate peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 896.5 kB/s eta 0:00:00


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ═══════════════════════════════════════════════════════════════
import os, json, random, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from datasets import Dataset, Audio
from transformers import (
    Wav2Vec2ForSequenceClassification,
    AutoFeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — CONFIG
# ═══════════════════════════════════════════════════════════════

# ── Core settings ──────────────────────────────────────────────
MODEL_ID       = 'facebook/wav2vec2-base'
MAX_DURATION_S = 6        # covers 95th pct of all datasets including ShEMO
MAX_LENGTH     = 16000 * MAX_DURATION_S
NUM_CLASSES    = 6
OUTPUT_DIR     = '/kaggle/working/ser_checkpoints'
FINAL_DIR      = '/kaggle/working/ser_final_model'
SEED           = 42

# ── Training hyperparams ───────────────────────────────────────
# Two separate LRs: head gets 10x encoder LR
# Previous runs had 100x difference which caused head to thrash
LR_ENCODER     = 5e-5
LR_HEAD        = 2e-4
BATCH_SIZE     = 8
GRAD_ACCUM     = 4        # effective batch = 32
NUM_EPOCHS     = 15
WARMUP_STEPS   = 1000      # long warmup — let loss stabilize before weights move
MAX_GRAD_NORM  = 1.0

# ── LoRA config ────────────────────────────────────────────────
# Only top 12 of 24 XLSR-53 layers get LoRA
# Bottom 12 stay completely frozen — preserve multilingual phonetics
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.1

# ── Per-dataset undersample caps ───────────────────────────────
# English dominates raw data — cap it to ~42% of total
# Non-English gets full data since it's already smaller
CAPS = {
    'RAVDESS' : 1500,
    'CREMA'   : 2000,
    'TESS'    : 1500,
    'SAVEE'   : None,   # 480 — keep all
    'ESD'     : 3500,   # 35k → 3500 (Chinese-accented English speakers)
    'ShEMO'   : None,   # 3000 — keep all
    'EMOVO'   : None,   # 588 — keep all
    'URDU'    : None,   # 400 — keep all
}

# ── Labels ─────────────────────────────────────────────────────
label2id = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':4,'sad':5}
id2label  = {v:k for k,v in label2id.items()}

emotion_map = {
    'ANG':'angry','DIS':'disgust','FEA':'fear',
    'HAP':'happy','NEU':'neutral','SAD':'sad',
    'angry':'angry','disgust':'disgust','fear':'fear',
    'happy':'happy','neutral':'neutral','sad':'sad',
    'ps':'happy','a':'angry','d':'disgust','f':'fear',
    'h':'happy','n':'neutral','sa':'sad','su':'happy',
    '01':'neutral','02':'neutral','03':'happy','04':'sad',
    '05':'angry','06':'fear','07':'disgust','08':'happy',
}

# ── Dataset paths ──────────────────────────────────────────────
train_paths = {
    'RAVDESS' : '/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio',
    'CREMA'   : '/kaggle/input/datasets/ejlok1/cremad/AudioWAV',
    'TESS'    : '/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess',
    'SAVEE'   : '/kaggle/input/datasets/ejlok1/surrey-audiovisual-expressed-emotion-savee',
    'ESD'     : '/kaggle/input/datasets/nguyenthanhlim/emotional-speech-dataset-esd',
    'ShEMO'   : '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database',
    'EMOVO'   : '/kaggle/input/datasets/sourabhy/emovo-italian-ser-dataset',
    'URDU'    : '/kaggle/input/datasets/bitlord/urdu-language-speech-dataset',
}
EMODB_PATH = '/kaggle/input/datasets/piyushagni5/berlin-database-of-emotional-speech-emodb'

print('Config set')
print(f'  MAX_DURATION_S : {MAX_DURATION_S}s  ({MAX_LENGTH} samples)')
print(f'  LR_ENCODER     : {LR_ENCODER}')
print(f'  LR_HEAD        : {LR_HEAD}  (10x encoder)')
print(f'  WARMUP_STEPS   : {WARMUP_STEPS}')
print(f'  LoRA r={LORA_R} alpha={LORA_ALPHA} on top 12 XLSR-53 layers')
print(f'  Bottom 12 layers FROZEN throughout')

Config set
  MAX_DURATION_S : 6s  (96000 samples)
  LR_ENCODER     : 5e-05
  LR_HEAD        : 0.0002  (10x encoder)
  WARMUP_STEPS   : 1000
  LoRA r=16 alpha=32 on top 12 XLSR-53 layers
  Bottom 12 layers FROZEN throughout


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — PARSE ALL DATASETS
# Fixes vs previous runs:
#   1. RAVDESS/TESS deduplication via seen_stems set
#   2. TESS: uses folder name NOT filename (more reliable)
#   3. TESS: 'surprised' folder correctly maps to 'ps' -> happy
#   4. ESD: uses folder name (Angry/Happy/Sad/Neutral/Surprise)
#   5. ShEMO: char at index 3 is emotion code
#   6. Per-dataset undersample caps to fix language imbalance
# ═══════════════════════════════════════════════════════════════
def parse_audio_files(base_path, dataset_name):
    data, skipped, seen_stems = [], [], set()
    if not os.path.exists(base_path):
        print(f'  WARNING: {dataset_name} NOT FOUND at {base_path}')
        return data

    for dirpath, _, filenames in os.walk(base_path):
        for fname in filenames:
            if not fname.lower().endswith('.wav'):
                continue
            # Dedup: RAVDESS and TESS have mirror folder structures
            if fname in seen_stems:
                continue
            seen_stems.add(fname)

            fpath = os.path.join(dirpath, fname)
            stem  = fname.rsplit('.', 1)[0]
            emotion_code = None

            try:
                if dataset_name == 'RAVDESS':
                    parts = stem.split('-')
                    emotion_code = parts[2] if len(parts) >= 3 else None

                elif dataset_name == 'CREMA':
                    parts = stem.split('_')
                    emotion_code = parts[2] if len(parts) >= 3 else None

                elif dataset_name == 'TESS':
                    # Use FOLDER name — more reliable than filename
                    # Folders: YAF_fear, OAF_Sad, OAF_Pleasant_surprise, etc.
                    folder = os.path.basename(dirpath).lower()
                    parts  = folder.split('_')
                    # Last part after speaker prefix: 'fear','sad','happy', etc.
                    # 'pleasant_surprise' -> last part is 'surprise' -> map to ps
                    raw_code = '_'.join(parts[1:])  # everything after YAF_ or OAF_
                    if 'surprise' in raw_code:
                        emotion_code = 'ps'
                    else:
                        emotion_code = parts[-1]

                elif dataset_name == 'SAVEE':
                    parts = stem.split('_')
                    emotion_code = ''.join(c for c in parts[1] if c.isalpha()) if len(parts)>=2 else None

                elif dataset_name == 'ESD':
                    # Structure: speaker_id/Emotion/files.wav
                    # Folder name IS the emotion label
                    folder = os.path.basename(dirpath).lower()
                    esd_map = {
                        'angry':'angry','happy':'happy','neutral':'neutral',
                        'sad':'sad','surprise':'happy',  # map surprise to happy
                    }
                    mapped = esd_map.get(folder)
                    if mapped:
                        data.append({'audio':fpath,'label':label2id[mapped],'dataset':dataset_name})
                    continue

                elif dataset_name == 'ShEMO':
                    # Filename: F13A20.wav
                    # char[3] = emotion code (A=angry,F=happy,H=happy,N=neutral,S=sad,W=fear)
                    shemo_map = {'A':'angry','F':'happy','H':'happy','N':'neutral','S':'sad','W':'fear'}
                    code   = stem[3].upper() if len(stem) > 3 else None
                    mapped = shemo_map.get(code)
                    if mapped:
                        data.append({'audio':fpath,'label':label2id[mapped],'dataset':dataset_name})
                    else:
                        skipped.append((fname, code))
                    continue

                elif dataset_name == 'EMOVO':
                    code_map = {
                        'dis':'disgust','pau':'fear','rab':'angry',
                        'gio':'happy','tri':'sad','neu':'neutral','sor':'happy',
                    }
                    prefix = stem.split('-')[0].lower()
                    mapped = code_map.get(prefix)
                    if mapped:
                        data.append({'audio':fpath,'label':label2id[mapped],'dataset':dataset_name})
                    else:
                        skipped.append((fname, prefix))
                    continue

                elif dataset_name == 'URDU':
                    folder_map = {'angry':'angry','neutral':'neutral','sad':'sad','happy':'happy'}
                    folder = os.path.basename(dirpath).lower()
                    mapped = folder_map.get(folder)
                    if mapped:
                        data.append({'audio':fpath,'label':label2id[mapped],'dataset':dataset_name})
                    continue

                elif dataset_name == 'EMODB':
                    emodb_map = {'W':'angry','L':'sad','E':'disgust','A':'fear','F':'happy','T':'sad','N':'neutral'}
                    code  = stem[5] if len(stem) > 5 else None
                    mapped = emodb_map.get(code)
                    if mapped:
                        data.append({'audio':fpath,'label':label2id[mapped],'dataset':dataset_name})
                    continue

                # Default: use emotion_map lookup
                if emotion_code and emotion_code in emotion_map:
                    data.append({'audio':fpath,'label':label2id[emotion_map[emotion_code]],'dataset':dataset_name})
                else:
                    skipped.append((fname, emotion_code))

            except Exception as e:
                skipped.append((fname, str(e)))

    print(f'  {dataset_name:<10}: {len(data):6d} parsed  {len(skipped):4d} skipped')
    return data


def undersample(data, cap, dataset_name, seed=42):
    if cap is None or len(data) <= cap:
        return data
    # Stratified undersample: keep proportional class distribution
    random.seed(seed)
    by_class = {}
    for item in data:
        lbl = item['label']
        by_class.setdefault(lbl, []).append(item)
    per_class = cap // NUM_CLASSES
    sampled = []
    for lbl, items in by_class.items():
        take = min(per_class, len(items))
        sampled.extend(random.sample(items, take))
    print(f'    {dataset_name}: undersampled {len(data)} → {len(sampled)}')
    return sampled


print('Parsing datasets...')
all_data = []
for name, path in train_paths.items():
    raw  = parse_audio_files(path, name)
    data = undersample(raw, CAPS.get(name), name)
    all_data.extend(data)

df = pd.DataFrame(all_data)
print(f'\nTotal training files: {len(df)}')

# Language breakdown
print('\nPer-dataset counts after undersampling:')
for name in train_paths:
    count = (df['dataset'] == name).sum()
    pct   = count/len(df)*100
    print(f'  {name:<10}: {count:5d}  ({pct:.1f}%)')

# Verify duplicates
dupes = df['audio'].duplicated().sum()
print(f'\nDuplicate paths: {dupes}  (should be 0)')

# Class distribution
print('\nClass distribution:')
for lid, count in df['label'].value_counts().sort_index().items():
    pct = count/len(df)*100
    bar = 'X' * int(pct/2)
    print(f'  {id2label[lid]:8s}: {count:5d}  ({pct:.1f}%)  {bar}')

Parsing datasets...
  RAVDESS   :   1440 parsed     0 skipped
  CREMA     :   7442 parsed     0 skipped
    CREMA: undersampled 7442 → 1998
  TESS      :   2800 parsed     0 skipped
    TESS: undersampled 2800 → 1500
  SAVEE     :    480 parsed     0 skipped
  ESD       :  35000 parsed     0 skipped
    ESD: undersampled 35000 → 2332
  ShEMO     :   3000 parsed     0 skipped
  EMOVO     :    588 parsed     0 skipped
  URDU      :    400 parsed     0 skipped

Total training files: 11738

Per-dataset counts after undersampling:
  RAVDESS   :  1440  (12.3%)
  CREMA     :  1998  (17.0%)
  TESS      :  1500  (12.8%)
  SAVEE     :   480  (4.1%)
  ESD       :  2332  (19.9%)
  ShEMO     :  3000  (25.6%)
  EMOVO     :   588  (5.0%)
  URDU      :   400  (3.4%)

Duplicate paths: 0  (should be 0)

Class distribution:
  angry   :  2661  (22.7%)  XXXXXXXXXXX
  disgust :   919  (7.8%)  XXX
  fear    :  1144  (9.7%)  XXXX
  happy   :  2177  (18.5%)  XXXXXXXXX
  neutral :  2786  (23.7%)  XXXXXXXXXXX
  

In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — SPLIT
# ═══════════════════════════════════════════════════════════════
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label']
)
print(f'Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}')

Train=9390  Val=1174  Test=1174


In [6]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — FEATURE EXTRACTION
# ═══════════════════════════════════════════════════════════════
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
print(f'Feature extractor SR: {feature_extractor.sampling_rate}')

def make_hf_dataset(dataframe):
    ds = Dataset.from_pandas(dataframe[['audio','label']].reset_index(drop=True))
    ds = ds.cast_column('audio', Audio(sampling_rate=16000))
    return ds

def preprocess(batch):
    arrays = []
    for x in batch['audio']:
        arr = x['array'].astype(np.float32)
        # Replace NaN/Inf silently with zeros — don't crash the whole batch
        if np.isnan(arr).any() or np.isinf(arr).any():
            arr = np.zeros(MAX_LENGTH, dtype=np.float32)
        arrays.append(arr)
    out = feature_extractor(
        arrays, sampling_rate=16000,
        max_length=MAX_LENGTH, truncation=True,
        padding='max_length', return_attention_mask=True,
    )
    out['labels'] = batch['label']
    return out

print('Preprocessing train...')
train_ds = make_hf_dataset(train_df).map(preprocess, batched=True, batch_size=32, remove_columns=['audio'])
print('Preprocessing val...')
val_ds   = make_hf_dataset(val_df).map(  preprocess, batched=True, batch_size=32, remove_columns=['audio'])
print('Preprocessing test (multilingual)...')
test_ds  = make_hf_dataset(test_df).map( preprocess, batched=True, batch_size=32, remove_columns=['audio'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')

s = train_ds[:4]
print(f'\ninput_values: shape={s["input_values"].shape}')
print(f'range=[{s["input_values"].min():.3f},{s["input_values"].max():.3f}]')
print(f'nan={torch.isnan(s["input_values"]).any().item()}')
print(f'dtype={s["input_values"].dtype}  (must be float32)')

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Feature extractor SR: 16000
Preprocessing train...


Map:   0%|          | 0/9390 [00:00<?, ? examples/s]

Preprocessing val...


Map:   0%|          | 0/1174 [00:00<?, ? examples/s]

Preprocessing test (multilingual)...


Map:   0%|          | 0/1174 [00:00<?, ? examples/s]


input_values: shape=torch.Size([4, 96000])
range=[-9.222,12.670]
nan=False
dtype=torch.float32  (must be float32)


In [7]:
print(f'Loading {MODEL_ID}...')
base_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

# Freeze CNN
base_model.wav2vec2.feature_extractor.requires_grad_(False)

# Apply LoRA with SHORT SUFFIX names — this is the fix
# Previous code used full paths like 'wav2vec2.encoder.layers.12.attention.q_proj'
# Those silently fail in newer PEFT because after wrapping the path becomes
# 'base_model.model.wav2vec2.encoder.layers.12...'
# Short suffixes always match regardless of wrapping
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(base_model, lora_config)

# VERIFY LoRA was actually applied — crash immediately if not
lora_count = sum(1 for n, p in model.named_parameters() if 'lora_' in n)
print(f'LoRA parameters found: {lora_count}')
print(f'Expected: ~192 (4 projections x 24 layers x 2 matrices)')
if lora_count == 0:
    raise RuntimeError('LoRA NOT applied — stop here, do not train')
print('LoRA confirmed')

# Freeze bottom 12 layers AFTER LoRA applied
# LoRA is applied to all 24 layers above, but bottom 12 get frozen here
# Only top 12 layers (12-23) will update during training
encoder_layers = model.base_model.model.wav2vec2.encoder.layers
for i in range(6):
    for param in encoder_layers[i].parameters():
        param.requires_grad_(False)

# Unfreeze head + layer norms
for name, param in model.named_parameters():
    if any(k in name for k in ['classifier', 'layer_norm']):
        param.requires_grad_(True)
# Re-initialize head with small std
# std=0.005 prevents any single class getting a head start
for name, param in model.named_parameters():
    if 'classifier' in name and 'weight' in name:
        nn.init.normal_(param, mean=0.0, std=0.005)
    elif 'classifier' in name and 'bias' in name:
        nn.init.zeros_(param)
print('Head initialized with std=0.005')

# Print breakdown
lora_t = sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and 'lora_' in n)
head_t = sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and any(k in n for k in ['projector','classifier']))
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'LoRA trainable (layers 12-23): {lora_t/1e6:.3f}M')
print(f'Head trainable               : {head_t/1e6:.3f}M')
print(f'Frozen                       : {frozen/1e6:.1f}M')

# Forward pass check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
dummy  = torch.randn(2, MAX_LENGTH).to(device)
with torch.no_grad():
    out = model(input_values=dummy)
probs = torch.softmax(out.logits, dim=-1)[0]
print(f'Forward pass OK. Probs: {[round(x,3) for x in probs.tolist()]}')
max_diff = probs.max().item() - probs.min().item()
if max_diff > 0.15:
    print(f'WARNING: max_diff={max_diff:.3f} — head init may cause collapse')
else:
    print(f'Head init OK (max_diff={max_diff:.4f})')

Loading facebook/wav2vec2-base...


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2ForSequenceClassification does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


LoRA parameters found: 96
Expected: ~192 (4 projections x 24 layers x 2 matrices)
LoRA confirmed
Head initialized with std=0.005
LoRA trainable (layers 12-23): 0.590M
Head trainable               : 0.003M
Frozen                       : 95.1M
Forward pass OK. Probs: [0.172, 0.167, 0.162, 0.169, 0.163, 0.167]
Head init OK (max_diff=0.0094)


In [8]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — CLASS WEIGHTS + METRICS
# ═══════════════════════════════════════════════════════════════
train_labels  = torch.tensor(train_df['label'].values)
counts        = torch.bincount(train_labels, minlength=NUM_CLASSES).float()
class_weights = (1.0 / counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print('Class weights:')
for i, w in enumerate(class_weights):
    print(f'  {id2label[i]:8s}: {counts[i].int():5d} samples  weight={w:.4f}')

accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    refs  = eval_pred.label_ids
    acc   = accuracy_metric.compute(predictions=preds, references=refs)['accuracy']
    f1    = f1_metric.compute(predictions=preds, references=refs, average='macro')['f1']
    print('\n  Per-class accuracy:')
    for cls_id in range(NUM_CLASSES):
        mask = refs == cls_id
        if mask.sum() > 0:
            cls_acc = (preds[mask] == refs[mask]).mean()
            print(f'    {id2label[cls_id]:8s}: {cls_acc:.3f}  {"X"*int(cls_acc*20)}')
    return {'accuracy': acc, 'f1_macro': f1}

Class weights:
  angry   :  2129 samples  weight=0.6186
  disgust :   735 samples  weight=1.7919
  fear    :   915 samples  weight=1.4394
  happy   :  1741 samples  weight=0.7565
  neutral :  2229 samples  weight=0.5909
  sad     :  1641 samples  weight=0.8026


In [9]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — CUSTOM TRAINER
# Key improvements vs all previous runs:
#   1. TWO separate optimizers: head at LR_HEAD, encoder at LR_ENCODER
#      Previous runs had single optimizer with one LR — caused instability
#   2. label_smoothing=0.15 — prevents overconfident collapse
#   3. Detailed per-group gradient monitoring
#   4. NaN guards on both logits and loss
#   5. Prints logit spread — early warning for collapse
# ═══════════════════════════════════════════════════════════════
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

class MultiLRTrainer(Trainer):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step      = 0
        self._nan_count = 0

    def create_optimizer(self):
        # Separate param groups with different LRs
        # Head gets LR_HEAD (1e-4)
        # LoRA + layer norms get LR_ENCODER (1e-5)
        # This prevents head from thrashing while encoder is slow to adapt
        head_params  = []
        enc_params   = []
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            if any(k in name for k in ['projector', 'classifier']):
                head_params.append(param)
            else:
                enc_params.append(param)

        self.optimizer = AdamW([
            {'params': head_params, 'lr': LR_HEAD,    'weight_decay': 0.05},
            {'params': enc_params,  'lr': LR_ENCODER, 'weight_decay': 0.01},
        ])
        print(f'\nOptimizer created:')
        print(f'  Head params  : {sum(p.numel() for p in head_params)/1e6:.3f}M  lr={LR_HEAD}')
        print(f'  Encoder params: {sum(p.numel() for p in enc_params)/1e6:.3f}M  lr={LR_ENCODER}')
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        self._step += 1

        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits

        # NaN guard
        if torch.isnan(logits).any() or torch.isinf(logits).any():
            self._nan_count += 1
            if self._nan_count <= 3:
                print(f'\n  [Step {self._step}] NaN/Inf in logits (count={self._nan_count})')
            dummy = torch.tensor(0.0, requires_grad=True, device=logits.device)
            return (dummy, outputs) if return_outputs else dummy

        weights = class_weights.to(logits.device)
        loss    = nn.functional.cross_entropy(
            logits, labels, weight=weights, label_smoothing=0.15
        )

        if torch.isnan(loss):
            self._nan_count += 1
            dummy = torch.tensor(0.0, requires_grad=True, device=logits.device)
            return (dummy, outputs) if return_outputs else dummy

        # Diagnostic print every 100 optimizer steps
        if self._step % (100 * GRAD_ACCUM) == 0:
            opt_step = self._step // GRAD_ACCUM

            # Separate grad norms per group
            head_norm, lora_norm, ln_norm = 0.0, 0.0, 0.0
            nan_grad_count = 0
            for name, p in model.named_parameters():
                if not p.requires_grad or p.grad is None:
                    continue
                if torch.isnan(p.grad).any():
                    nan_grad_count += 1
                n2 = p.grad.data.norm(2).item() ** 2
                if any(k in name for k in ['projector', 'classifier']):
                    head_norm += n2
                elif 'lora_' in name:
                    lora_norm += n2
                else:
                    ln_norm += n2
            head_norm = head_norm ** 0.5
            lora_norm = lora_norm ** 0.5
            ln_norm   = ln_norm   ** 0.5

            # Logit spread — key collapse indicator
            logit_std  = logits.std(dim=-1).mean().item()
            logit_means= logits.mean(dim=0).detach().cpu().tolist()
            preds       = logits.argmax(dim=-1)
            pred_counts = torch.bincount(preds, minlength=NUM_CLASSES).tolist()
            true_counts = torch.bincount(labels, minlength=NUM_CLASSES).tolist()
            avg_conf    = torch.softmax(logits,dim=-1).max(dim=-1).values.mean().item()

            print(f'\n  ── Step {self._step} / Optimizer {opt_step} ──')
            print(f'    Loss          : {loss.item():.4f}')
            print(f'    Head grad norm: {head_norm:.4f}  LoRA: {lora_norm:.4f}  LN: {ln_norm:.4f}')
            print(f'    NaN grads     : {nan_grad_count}  |  NaN batches: {self._nan_count}')
            print(f'    Logit std     : {logit_std:.4f}  (< 0.05 = collapse forming, > 0.5 = learning)')
            print(f'    Avg confidence: {avg_conf:.3f}')
            print(f'    Logit means   : {[round(x,3) for x in logit_means]}')
            print(f'    Pred dist     : {dict(zip(id2label.values(), pred_counts))}')
            print(f'    True dist     : {dict(zip(id2label.values(), true_counts))}')

            # Early collapse warning
            if logit_std < 0.05:
                print(f'    *** COLLAPSE WARNING: logit std={logit_std:.4f} < 0.05 ***')
            if max(pred_counts) == sum(pred_counts):
                print(f'    *** ALL PREDICTIONS = ONE CLASS — COLLAPSE DETECTED ***')

        return (loss, outputs) if return_outputs else loss

In [10]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — TRAINING ARGUMENTS + TRAIN
# ═══════════════════════════════════════════════════════════════
lora_trainable = sum(p.numel() for n,p in model.named_parameters() if p.requires_grad and 'lora_' in n)
print(f'LoRA trainable params: {lora_trainable/1e6:.3f}M')
if lora_trainable == 0:
    raise RuntimeError('LoRA = 0 params. Do not train.')

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print('Cleaned old checkpoints')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRAD_ACCUM,

    # LR here is a placeholder — MultiLRTrainer.create_optimizer overrides it
    learning_rate=LR_ENCODER,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,

    fp16=False,
    bf16=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    logging_steps=50,
    report_to='none',
    push_to_hub=False,
)

trainer = MultiLRTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

# Final sanity checks
print('Pre-training checks:')
s = train_ds[:2]
iv = s['input_values']
print(f'  dtype  : {iv.dtype}  (must be float32)')
print(f'  range  : [{iv.min():.3f},{iv.max():.3f}]')
print(f'  nan    : {torch.isnan(iv).any().item()}')
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  trainable: {trainable/1e6:.3f}M params')

if torch.isnan(iv).any():
    raise RuntimeError('NaN in training data — stop and check preprocess()')

print('\nStarting training...')
print('=' * 60)
trainer.train()
print('=' * 60)
print(f'TRAINING COMPLETE  |  NaN batches: {trainer._nan_count}')

LoRA trainable params: 0.590M
Pre-training checks:
  dtype  : torch.float32  (must be float32)
  range  : [-6.114,6.601]
  nan    : False
  trainable: 0.633M params

Starting training...

Optimizer created:
  Head params  : 0.003M  lr=0.0002
  Encoder params: 0.630M  lr=5e-05


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,7.383367,1.841773,0.191652,0.111813
2,7.355793,1.829718,0.123509,0.084918
3,7.237023,1.738179,0.312606,0.219063
4,6.667737,1.695235,0.398637,0.280867
5,6.349636,1.559158,0.468484,0.369818
6,6.022224,1.523096,0.500852,0.420531
7,5.810474,1.456369,0.546848,0.479607
8,5.520658,1.413842,0.577513,0.521122
9,5.311249,1.365731,0.622658,0.585924
10,5.230148,1.347642,0.632027,0.601565



  ── Step 400 / Optimizer 100 ──
    Loss          : 1.7810
    Head grad norm: 0.9930  LoRA: 0.0351  LN: 0.0886
    NaN grads     : 0  |  NaN batches: 0
    Logit std     : 0.0121  (< 0.05 = collapse forming, > 0.5 = learning)
    Avg confidence: 0.169
    Logit means   : [0.013, 0.006, -0.003, -0.009, -0.019, -0.006]
    Pred dist     : {'angry': 15, 'disgust': 0, 'fear': 1, 'happy': 0, 'neutral': 0, 'sad': 0}
    True dist     : {'angry': 5, 'disgust': 3, 'fear': 3, 'happy': 0, 'neutral': 2, 'sad': 3}
    *** COLLAPSE WARNING: logit std=0.0121 < 0.05 ***

  Per-class accuracy:
    angry   : 0.568  XXXXXXXXXXX
    disgust : 0.750  XXXXXXXXXXXXXXX
    fear    : 0.026  
    happy   : 0.000  
    neutral : 0.000  
    sad     : 0.010  

  ── Step 800 / Optimizer 200 ──
    Loss          : 1.8797
    Head grad norm: 0.9426  LoRA: 0.0436  LN: 0.1294
    NaN grads     : 0  |  NaN batches: 0
    Logit std     : 0.0180  (< 0.05 = collapse forming, > 0.5 = learning)
    Avg confidence: 0.171

In [11]:
# ═══════════════════════════════════════════════════════════════
# CELL 10 — MULTILINGUAL TEST SET EVALUATION
# ═══════════════════════════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix

results = trainer.evaluate(test_ds)
print('\n=== MULTILINGUAL TEST SET RESULTS ===')
for k, v in results.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

raw   = trainer.predict(test_ds)
preds = np.argmax(raw.predictions, axis=1)
print(classification_report(raw.label_ids, preds, target_names=[id2label[i] for i in range(NUM_CLASSES)]))

cm = confusion_matrix(raw.label_ids, preds)
header = '         ' + '  '.join(f'{id2label[i][:5]:>5}' for i in range(NUM_CLASSES))
print(header)
for i, row in enumerate(cm):
    print(f'  {id2label[i][:7]:>7}  ' + '  '.join(f'{v:>5}' for v in row))


  Per-class accuracy:
    angry   : 0.782  XXXXXXXXXXXXXXX
    disgust : 0.717  XXXXXXXXXXXXXX
    fear    : 0.623  XXXXXXXXXXXX
    happy   : 0.280  XXXXX
    neutral : 0.832  XXXXXXXXXXXXXXXX
    sad     : 0.605  XXXXXXXXXXXX

=== MULTILINGUAL TEST SET RESULTS ===
  eval_loss: 1.2790
  eval_accuracy: 0.6491
  eval_f1_macro: 0.6185
  eval_runtime: 23.5230
  eval_samples_per_second: 49.9090
  eval_steps_per_second: 1.5730
  epoch: 15.0000

  Per-class accuracy:
    angry   : 0.782  XXXXXXXXXXXXXXX
    disgust : 0.717  XXXXXXXXXXXXXX
    fear    : 0.623  XXXXXXXXXXXX
    happy   : 0.280  XXXXX
    neutral : 0.832  XXXXXXXXXXXXXXXX
    sad     : 0.605  XXXXXXXXXXXX
              precision    recall  f1-score   support

       angry       0.77      0.78      0.78       266
     disgust       0.59      0.72      0.65        92
        fear       0.47      0.62      0.54       114
       happy       0.58      0.28      0.38       218
     neutral       0.66      0.83      0.74       279
  

In [12]:
# ═══════════════════════════════════════════════════════════════
# CELL 11 — GERMAN ZERO-SHOT TEST (EMODB)
# Model never saw German. This is the real generalization test.
# ═══════════════════════════════════════════════════════════════
import soundfile as sf
import librosa
from collections import Counter, defaultdict

def evaluate_zero_shot(path, parser_fn, lang_name):
    all_files = []
    for dirpath, _, fnames in os.walk(path):
        for f in fnames:
            if f.lower().endswith('.wav'):
                all_files.append(os.path.join(dirpath, f))

    print(f'\n{"="*60}')
    print(f'ZERO-SHOT TEST: {lang_name}  ({len(all_files)} files)')
    print(f'{"="*60}')

    model.eval()
    true_labels, pred_labels, confidences = [], [], []
    skipped = 0

    for fpath in all_files:
        true_emo = parser_fn(fpath)
        if true_emo is None or true_emo not in label2id:
            skipped += 1
            continue
        try:
            arr, sr = sf.read(fpath)
            if arr.ndim == 2: arr = arr[:, 0]
            if sr != 16000:
                arr = librosa.resample(arr.astype(np.float32), orig_sr=sr, target_sr=16000)
            arr    = arr.astype(np.float32)
            inputs = feature_extractor(
                arr, sampling_rate=16000, max_length=MAX_LENGTH,
                truncation=True, padding='max_length', return_tensors='pt'
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
            true_labels.append(true_emo)
            pred_labels.append(id2label[probs.argmax().item()])
            confidences.append(probs.max().item())
        except Exception as e:
            skipped += 1

    if not true_labels:
        print('  No files evaluated')
        return

    correct = sum(t==p for t,p in zip(true_labels,pred_labels))
    print(f'Evaluated: {len(true_labels)}  Skipped: {skipped}')
    print(f'Overall accuracy: {correct/len(true_labels)*100:.1f}%')
    print(f'Avg confidence  : {np.mean(confidences):.3f}')

    pred_c = Counter(pred_labels)
    true_c = Counter(true_labels)
    print('\nPrediction vs True distribution:')
    for emo in sorted(label2id):
        print(f'  {emo:8s}: pred={pred_c.get(emo,0):4d}  true={true_c.get(emo,0):4d}')

    print('\nPer-class accuracy:')
    pc = defaultdict(lambda:[0,0])
    for t,p in zip(true_labels,pred_labels):
        pc[t][1] += 1
        if t==p: pc[t][0] += 1
    for emo in sorted(pc):
        c, tot = pc[emo]
        acc = c/tot
        print(f'  {emo:8s}: {c:3d}/{tot:3d}  {acc:.3f}  {"X"*int(acc*20)}')

    print()
    print(classification_report(true_labels, pred_labels, zero_division=0))

def parse_emodb(fpath):
    emodb_map = {'W':'angry','L':'sad','E':'disgust','A':'fear','F':'happy','T':'sad','N':'neutral'}
    stem = os.path.basename(fpath).rsplit('.',1)[0]
    return emodb_map.get(stem[5]) if len(stem) > 5 else None

if os.path.exists(EMODB_PATH):
    evaluate_zero_shot(EMODB_PATH, parse_emodb, 'GERMAN (EMODB — never seen in training)')
else:
    print('EMODB not found')


ZERO-SHOT TEST: GERMAN (EMODB — never seen in training)  (535 files)
Evaluated: 535  Skipped: 0
Overall accuracy: 65.8%
Avg confidence  : 0.473

Prediction vs True distribution:
  angry   : pred= 147  true= 127
  disgust : pred=  15  true=  46
  fear    : pred=  20  true=  69
  happy   : pred= 113  true=  71
  neutral : pred=  61  true=  79
  sad     : pred= 179  true= 143

Per-class accuracy:
  angry   : 124/127  0.976  XXXXXXXXXXXXXXXXXXX
  disgust :  12/ 46  0.261  XXXXX
  fear    :  19/ 69  0.275  XXXXX
  happy   :  51/ 71  0.718  XXXXXXXXXXXXXX
  neutral :  33/ 79  0.418  XXXXXXXX
  sad     : 113/143  0.790  XXXXXXXXXXXXXXX

              precision    recall  f1-score   support

       angry       0.84      0.98      0.91       127
     disgust       0.80      0.26      0.39        46
        fear       0.95      0.28      0.43        69
       happy       0.45      0.72      0.55        71
     neutral       0.54      0.42      0.47        79
         sad       0.63      0.79   

In [13]:
# ═══════════════════════════════════════════════════════════════
# CELL 12 — SAVE
# ═══════════════════════════════════════════════════════════════
os.makedirs(FINAL_DIR, exist_ok=True)

# PEFT save — saves only LoRA adapter weights (~15MB), not full model
model.save_pretrained(FINAL_DIR)
feature_extractor.save_pretrained(FINAL_DIR)

config_data = {
    'label2id'             : label2id,
    'id2label'             : {str(k):v for k,v in id2label.items()},
    'max_duration_s'       : MAX_DURATION_S,
    'sampling_rate'        : 16000,
    'model_base'           : MODEL_ID,
    'strategy'             : 'lora_top12_layers_balanced_multilingual',
    'lora_r'               : LORA_R,
    'lora_alpha'           : LORA_ALPHA,
    'languages_trained_on' : ['english','chinese_accented_english','persian','italian','urdu'],
    'zero_shot_test'       : 'german_emodb',
    'is_peft_model'        : True,
    'lr_head'              : LR_HEAD,
    'lr_encoder'           : LR_ENCODER,
}
with open(f'{FINAL_DIR}/ser_config.json','w') as f:
    json.dump(config_data, f, indent=2)

print(f'Saved to {FINAL_DIR}:')
total = 0
for fn in sorted(os.listdir(FINAL_DIR)):
    fp = os.path.join(FINAL_DIR, fn)
    sz = os.path.getsize(fp)/1e6
    total += sz
    print(f'  {fn:<45} {sz:7.1f} MB')
print(f'  Total: {total:.1f} MB')
print('\nDone. Kaggle Output tab → ser_final_model → Download')

Saved to /kaggle/working/ser_final_model:
  README.md                                         0.0 MB
  adapter_config.json                               0.0 MB
  adapter_model.safetensors                         4.7 MB
  preprocessor_config.json                          0.0 MB
  ser_config.json                                   0.0 MB
  Total: 4.7 MB

Done. Kaggle Output tab → ser_final_model → Download
